#Act 1 - DatFrame CSV + Spark Samples

In [0]:
#!pip install pandas
#!pip install matplotlib
#!pip install numpy  

In [0]:
#Libraries
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [0]:
#Sample CSV file from JornadaDeDados codespace convert to spark df (big data)

#Clients
df_sparked_clients = spark.read.csv("/Workspace/Users/luizedf98@hotmail.com/data-engineering-roadmap/00-imersao-jornada/data/clientes.csv", header=True, inferSchema=True)
df_clients = df_sparked_clients.toPandas()

#Orders
df_sparked_orders = spark.read.csv("/Workspace/Users/luizedf98@hotmail.com/data-engineering-roadmap/00-imersao-jornada/data/vendas.csv", header=True, inferSchema=True)
df_orders = df_sparked_orders.toPandas()


In [0]:
#Check clients
display(df_clients)

In [0]:
#Call df_orders + create new cols
df_orders['total_preco'] = round(df_orders['quantidade'] * df_orders['preco_unitario'],2)

display(df_orders)

In [0]:
#Merge dfs
df_merged_clients_orders = pd.merge(df_clients, df_orders, on='id_cliente', how='left')
#display(df_merged_clients_orders)

#Group by estado + canal_venda
df_grouped = df_merged_clients_orders.groupby(['estado','canal_venda']).agg({'total_preco': 'sum'}).reset_index()
df_grouped = df_grouped.sort_values(
    by=['estado', 'total_preco'],
    ascending=[True, False]
)
display(df_grouped)

In [0]:
#Create stacked bar graph with df_grouped by estado and canal_venda per Total de vendas
df_grouped_pivot = df_grouped.pivot(index='estado', columns='canal_venda', values='total_preco').fillna(0)
df_grouped_pivot = df_grouped_pivot.reindex(
    sorted(df_grouped_pivot.columns, reverse=True),
    axis=1
)
df_grouped_pivot.plot(kind='bar', stacked=True,figsize=(15, 10))

#Show data values on top of each bar
for i, col in enumerate(df_grouped_pivot.columns):
    for j, val in enumerate(df_grouped_pivot[col]):
        plt.text(
            j,
            val+2000,
            f'R$ {val:,.2f}'.replace(',', 'X').replace('.', ',').replace('X', '.'),
            ha='center',
            va='bottom',
            rotation=0
        )
plt.title('Total de vendas por estado e canal de venda')
plt.xlabel('Estado')
plt.ylabel('Total de vendas')
plt.legend(title='Canal de venda')
plt.show()
